In [2]:
from pathlib import Path
from sklearn.metrics import confusion_matrix
from scripts.data_processing.storage import load_parquet_df
import pandas as pd
import numpy as np
import requests

In [3]:
players_df = load_parquet_df("silver_players.parquet")
matches_df = load_parquet_df("silver_matches.parquet")
matches_df.columns, players_df.columns

(Index(['match_id', 'start_time', 'winning_team', 'duration_s', 'match_outcome',
        'average_badge', 'rewards_eligible'],
       dtype='object'),
 Index(['account_id', 'hero_id', 'player_slot', 'team', 'kills', 'deaths',
        'assists', 'net_worth', 'last_hits', 'denies', 'ability_points',
        'assigned_lane', 'player_level', 'player_match_outcome',
        'final_stats.ability_kills', 'final_stats.ability_points',
        'final_stats.absorption_provided', 'final_stats.assists',
        'final_stats.boss_damage', 'final_stats.bullet_kills',
        'final_stats.creep_damage', 'final_stats.creep_kills',
        'final_stats.damage_absorbed', 'final_stats.damage_mitigated',
        'final_stats.deaths', 'final_stats.denies', 'final_stats.gold_boss',
        'final_stats.gold_boss_orb', 'final_stats.gold_death_loss',
        'final_stats.gold_denied', 'final_stats.gold_lane_creep',
        'final_stats.gold_lane_creep_orbs', 'final_stats.gold_neutral_creep',
        'final_st

A17. Statistical power analysis ●●○ Applied · ~2 hrs

Task: Before running a test on new data, compute the sample size needed to detect a given effect size at 80% power, using statsmodels.stats.power. Done: You validate the formula by simulation — simulate the test many times at your computed sample size on data with the assumed true effect, and confirm empirical power lands near 80%. Watch for: Underpowered studies don't just miss real effects — they systematically overestimate effect size in the ones that do come back significant. Know this before running a real experiment.

In [4]:
players_df["match_outcome_binary"] = players_df["player_match_outcome"].map({"Win": 1, "Loss": 0})

BASE = "https://api.deadlock-api.com"
HEADERS = {"User-Agent": "python-requests (deadlock-exploration-script)"}

r = requests.get(f"{BASE}/v1/assets/heroes", headers=HEADERS, timeout=30)
r.raise_for_status()
hero_names = pd.DataFrame(r.json())[["id", "name"]].rename(columns={"id": "hero_id", "name": "hero_name"})

players_df = players_df.merge(hero_names, on="hero_id", how="left")

In [7]:
players_df.columns
prop2 = players_df["match_outcome_binary"].mean()

prop1 = prop2 + 0.02


In [12]:
import math
from statsmodels.stats.power import tt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize


players_df.columns
prop2 = players_df["match_outcome_binary"].mean()

prop1 = prop2 + 0.40

effect_size = proportion_effectsize(prop1=prop1, prop2=prop2)



# Alpha is the chance of noise, power is the probalbity you can reject the null hypothesis
sample_size_per_group = tt_ind_solve_power(
    effect_size=effect_size,
    nobs1=None,
    alpha=0.5,
    power=0.80,
    ratio=1.0
)

required_n = math.ceil(sample_size_per_group)

print(f"Current Win Rate (Baseline): {prop2:.4f}")
print(f"Target Win Rate: {prop1:.4f}")
print(f"Calculated Effect Size: {effect_size:.4f}")
print(f"Required sample size per group (e.g., Treatment vs Control): {required_n}")

Current Win Rate (Baseline): 0.5090
Target Win Rate: 0.9090
Calculated Effect Size: 0.9399
Required sample size per group (e.g., Treatment vs Control): 5


In [13]:
import pandas as pd

# 1. Map the internal numbers to real map locations
lane_names = {1: "Left Lane", 4: "Middle Lane", 6: "Right Lane"}
players_df["lane_name"] = players_df["assigned_lane"].map(lane_names)

# 2. Group by hero and lane to calculate match counts and win rates
grouped = players_df.groupby(["hero_name", "lane_name"])["match_outcome_binary"]
grouped_data = grouped.agg(games_played="count", win_rate="mean").reset_index()

# 3. Filter out noise (only keep combinations with at least 30 matches)
# Adjust or remove this threshold depending on your total dataset size
min_games_threshold = 30
filtered_data = grouped_data[grouped_data["games_played"] >= min_games_threshold]

# 4. Pivot the data into a clean matrix
win_rate_matrix = filtered_data.pivot(
    index="hero_name", columns="lane_name", values="win_rate"
)

# 5. Arrange columns from left to right across the map
columns_order = ["Left Lane", "Middle Lane", "Right Lane"]
win_rate_matrix = win_rate_matrix.reindex(columns=columns_order)

# 6. Format as clean percentages and mark unplayed/filtered lanes as "N/A"
formatted_matrix = win_rate_matrix.map(
    lambda x: f"{x * 100:.1f}%" if pd.notnull(x) else "N/A"
)

# 7. Print the final dashboard
print("\n--- DEADLOCK HERO WIN RATES BY ASSIGNED LANE ---")
print(formatted_matrix)

# Check how many actual matches are driving these weird percentages
counts_matrix = filtered_data.pivot(
    index="hero_name", columns="lane_name", values="games_played"
)
print(counts_matrix.reindex(columns=["Left Lane", "Middle Lane", "Right Lane"]))


--- DEADLOCK HERO WIN RATES BY ASSIGNED LANE ---
lane_name   Left Lane Middle Lane Right Lane
hero_name                                   
Abrams          45.1%       53.9%      57.6%
Apollo          66.7%       52.1%      61.9%
Bebop           49.4%       49.5%      43.2%
Billy           47.2%       56.2%      48.5%
Calico          49.0%       47.6%      53.8%
Celeste         54.1%       67.2%      60.0%
Drifter         49.3%       51.0%      53.4%
Dynamo          51.2%       58.0%      62.1%
Graves          64.7%       51.0%      57.8%
Grey Talon      56.2%       62.3%      62.3%
Haze            51.7%       50.7%      49.2%
Holliday        52.9%       54.0%      42.9%
Infernus        50.3%       40.1%      52.6%
Ivy             50.7%       45.2%      49.1%
Kelvin          50.8%       59.5%      54.5%
Lady Geist      58.6%       64.5%      50.0%
Lash            53.7%       40.6%      50.5%
McGinnis        51.0%       52.4%      63.6%
Mina            49.1%       51.3%      44.8%
Mirag